# JAVDL — Google Colab
Clean, ordered test/runner notebook for the `javdl` branch.

**Run the cells from top to bottom after a fresh Colab runtime.**

**Required Colab Secrets:** TELEGRAM_API_ID, TELEGRAM_API_HASH, TELEGRAM_BOT_TOKEN.

Secrets are loaded at runtime and are never written into the repository.


## 1. Clone the javdl branch


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO = "https://github.com/Telebotfaroff/javtiful-scraper.git"
BRANCH = "javdl"
WORKDIR = Path("/content/javdl")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(WORKDIR)],
    check=True,
)
os.chdir(WORKDIR)
print("Repository:", REPO)
print("Branch:", BRANCH)
print("Working directory:", Path.cwd())


## 2. Install system and Python dependencies


In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg git
!python -m pip install -q -r requirements.txt
!python -m playwright install --with-deps chromium


## 3. Load Telegram credentials from Colab Secrets

In Colab, open Secrets and add the three required values before running this cell.


In [ ]:
from google.colab import userdata

def get_secret(name):
    try:
        return userdata.get(name)
    except Exception as exc:
        raise RuntimeError(f"Unable to read Colab Secret {name!r}: {exc}") from exc

TELEGRAM_API_ID = get_secret("TELEGRAM_API_ID")
TELEGRAM_API_HASH = get_secret("TELEGRAM_API_HASH")
TELEGRAM_BOT_TOKEN = get_secret("TELEGRAM_BOT_TOKEN")

missing = [
    name for name, value in {
        "TELEGRAM_API_ID": TELEGRAM_API_ID,
        "TELEGRAM_API_HASH": TELEGRAM_API_HASH,
        "TELEGRAM_BOT_TOKEN": TELEGRAM_BOT_TOKEN,
    }.items()
    if not value
]

if missing:
    raise RuntimeError("Missing Colab Secrets: " + ", ".join(missing))

os.environ.update({
    "TELEGRAM_API_ID": str(TELEGRAM_API_ID),
    "TELEGRAM_API_HASH": str(TELEGRAM_API_HASH),
    "TELEGRAM_BOT_TOKEN": str(TELEGRAM_BOT_TOKEN),
    "TELEGRAM_SESSION": "javdl_colab",
    "TELEGRAM_UPLOAD_SESSION": "javdl_uploads",
    "TELEGRAM_MAX_CONCURRENT_TRANSMISSIONS": "2",
    "DEFAULT_UPLOADER": "gofile",
})

print("Telegram credentials loaded from Colab Secrets.")
print("Dedicated upload session: javdl_uploads")
print("Telegram upload concurrency: 2")


## 4. Environment check


In [ ]:
import shutil
import subprocess
import sys

try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True,
        stderr=subprocess.DEVNULL,
    ).strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    gpu = "No NVIDIA GPU detected"

print("GPU:", gpu or "No NVIDIA GPU detected")
print("FFmpeg:", shutil.which("ffmpeg") or "NOT FOUND")
print("FFprobe:", shutil.which("ffprobe") or "NOT FOUND")
print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())


## 5. Import and syntax test


In [ ]:
!python -m compileall -q app main.py scripts/colab.py

from app.extractor.javtiful import JavtifulExtractor
from app.uploaders.manager import UploadManager

extractor = JavtifulExtractor()
uploads = UploadManager()

print("Extractor:", extractor.name)
print("Upload providers:", uploads.names())


## 6. Test GoFile guest uploader

This uploads a tiny local test file and verifies that GoFile returns a download URL. It does not involve Telegram.


In [ ]:
from pathlib import Path

test_file = Path("/content/javdl_gofile_test.txt")
test_file.write_text("JAVDL GoFile guest-upload test", encoding="utf-8")

gofile = uploads.get("gofile")
try:
    result = gofile.upload(test_file)
    print("Provider:", result["provider"])
    print("URL:", result.get("download_url"))
    print("Verified:", gofile.verify(result))
finally:
    test_file.unlink(missing_ok=True)


## 7. Test Javtiful stream extraction

Set a Javtiful post URL you are authorized to access. This checks the guide-based extraction order: `#frontWatchConfig.playerSources`, inline `playerSources`, DOM `<video>/<source>`, embedded media URLs, then runtime browser detection.


In [ ]:
JAVTIFUL_URL = ""

if not JAVTIFUL_URL.strip():
    print("Set JAVTIFUL_URL and run this cell to test extraction.")
else:
    video = extractor.extract(JAVTIFUL_URL.strip())
    print("Title:", video.title)
    print("Duration:", video.duration)
    print("Thumbnail:", video.thumbnail)
    print("Qualities:", list(video.qualities))
    for quality, stream_url in video.qualities.items():
        print(f"{quality}: {stream_url}")
    if not video.qualities:
        raise RuntimeError("No downloadable stream/source URL was exposed by the page.")


## 8. Test stream download

Optional direct download test. Set `DOWNLOAD_TEST = True` after extraction succeeds. This downloads the selected stream with the Javtiful page as the `Referer`.


In [ ]:
DOWNLOAD_TEST = False
DOWNLOAD_QUALITY = "best"

if not DOWNLOAD_TEST:
    print("Set DOWNLOAD_TEST = True to download the selected stream.")
elif not JAVTIFUL_URL.strip():
    raise RuntimeError("Set JAVTIFUL_URL in the extraction cell first.")
else:
    from app.downloader.downloader import Downloader

    if not video.qualities:
        raise RuntimeError("No stream URL available for download.")

    downloader = Downloader()
    path = downloader.download(video, DOWNLOAD_QUALITY)
    print("Selected quality:", video.selected_quality)
    print("Downloaded:", path)
    print("Size (MB):", round(Path(path).stat().st_size / 1024 / 1024, 2))


In [ ]:
#@title 9. Enable detailed live logs
import logging
import sys

# Send application logs directly to the Colab output so extraction,
# thumbnail, FFmpeg/ffprobe, download and Telegram upload diagnostics are visible.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    stream=sys.stdout,
    force=True,
)

for logger_name in ("app", "app.extractor", "app.downloader", "app.uploaders", "app.jobs", "app.bot"):
    logging.getLogger(logger_name).setLevel(logging.INFO)

print("Detailed application logging enabled.")
print("Upload test configuration:")
print("  Session:", os.environ.get("TELEGRAM_UPLOAD_SESSION"))
print("  max_concurrent_transmissions:", os.environ.get("TELEGRAM_MAX_CONCURRENT_TRANSMISSIONS"))
print("Watch the output for THUMBNAIL:, TELEGRAM UPLOAD:, PIPELINE:, FFmpeg and ffprobe diagnostics.")


## 10. Telegram upload performance test

This test uses the dedicated Pyrogram upload session configured above. `max_concurrent_transmissions` is set to `2` for this experiment.

**Expected log:** `TELEGRAM UPLOAD: dedicated session=javdl_uploads max_concurrent_transmissions=2`.

The uploader client stays connected between jobs. Do not change the concurrency value during this test; compare the measured upload speed against the previous `javdl` run first.


## 11. Start the Telegram bot

**Colab/Jupyter note:** do not use `app.run()`. Colab already has an asyncio event loop. This cell starts Pyrogram asynchronously and waits for it safely.

After starting, send the bot a Javtiful URL. The test flow is:
1. Extract metadata and actual stream/quality URLs.
2. Choose a quality from the inline buttons.
3. Choose **Download Full Video** or **Download Clip**.
4. For clip mode, send one range such as `00:00-01:30`.
5. The pipeline downloads the selected stream, optionally clips it, uploads it to Telegram, and reports progress/errors.

Run this cell only after the previous checks pass. Stop it with the Colab interrupt/stop control.


In [ ]:
from pyrogram import idle
from app.bot.telegram import create_app

app = create_app()
print("Starting JAVDL Telegram bot…")

await app.start()
print("JAVDL Telegram bot is running. Send a Javtiful URL to the bot.")

try:
    await idle()
finally:
    if app.is_connected:
        await app.stop()
    print("JAVDL Telegram bot stopped.")


In [ ]:
#@title 12. Inspect saved application log (after stopping the bot)
from pathlib import Path

candidates = [Path("data/javdl.log"), Path("data/app.log"), Path("javdl.log")]
found = [p for p in candidates if p.exists()]
if not found:
    print("No file-based application log was found; use the Colab output above.")
else:
    for log_path in found:
        print(f"\n===== {log_path} =====")
        print(log_path.read_text(encoding="utf-8", errors="replace")[-20000:])
